# Same shortlist as JSON

Give me that same S&P 500 shortlist and coverage as JSON, keeping the original order and snapshot.

**Data and assumptions**

- Use the completed screen linked by the notebook, with the original membership, price cutoff and exclusions rather than a refreshed screen.

**Requested answer:** screener_json (json).

This notebook contains its complete request and setup. Share this file; no local case JSON or helper module is required. It contains no pre-filled market results.

First complete [04_sp500_momentum_screen.ipynb](04_sp500_momentum_screen.ipynb). This follow-up reads that saved operation and uses the same execution. It does not start the prerequisite automatically.

## Connect

Select a Python kernel with `epoch-sdk[notebook]` installed. On the prepared
server, choose **Epoch SDK (local prototype)**.

The notebook needs only your Epoch login and the two service addresses. The
collapsed helper below obtains a current identity token from the environment's
configured Epoch login command, or uses `EPOCH_TOKEN`. It prints no credentials
and reads no service configuration files. On another machine, supply your own
`EPOCH_TOKEN`, `EPOCH_BACKEND_URL` and `EPOCH_GRAPH_URL`.

The next cell prepares the client and checks both services. It starts no research.

In [ ]:
"""Obtain only an Epoch bearer token from the local operator's login provider."""

from __future__ import annotations

import json
import os
import subprocess


def identity_token() -> str:
    """Read a static token or ask the configured login helper for a current one.

    The prepared notebook launcher configures a trusted argv list in the
    environment. It uses the owner's normal Epoch login/refresh flow. No shell
    is used, and neither provider output nor credentials appear in exceptions.
    Standalone notebook environments can continue to supply EPOCH_TOKEN alone.
    """
    command = os.environ.get("EPOCH_NOTEBOOK_TOKEN_COMMAND")
    if command:
        try:
            argv = json.loads(command)
        except (TypeError, ValueError):
            raise RuntimeError(
                "The notebook's Epoch login command is invalid."
            ) from None
        if (
            not isinstance(argv, list)
            or not argv
            or not all(isinstance(value, str) and value for value in argv)
        ):
            raise RuntimeError(
                "The notebook's Epoch login command must be a JSON argv list."
            )
        try:
            result = subprocess.run(
                argv, capture_output=True, text=True, timeout=30, check=False
            )
        except (OSError, subprocess.TimeoutExpired):
            raise RuntimeError(
                "Could not contact the local Epoch login helper. Restore the normal "
                "Epoch login and rerun this setup cell."
            ) from None
        token = result.stdout.strip()
        if result.returncode or not token or any(value.isspace() for value in token):
            raise RuntimeError(
                "The Epoch login needs attention. Sign in through the normal login "
                "flow, then rerun this setup cell."
            )
        return token
    token = os.environ.get("EPOCH_TOKEN")
    if not token:
        raise RuntimeError(
            "Set EPOCH_TOKEN to your Epoch identity token, or use the prepared local "
            "notebook server, which supplies Epoch login to its kernels."
        )
    return token

In [ ]:
import importlib.metadata
import httpx
from IPython.display import JSON, display
from epoch_sdk import RunRequest
from epoch_sdk.research import EpochClient, ResearchCompleted

# Connect to Epoch using your login, not vendor or model credentials.
_epoch_token = identity_token()
epoch = EpochClient.connect(
    graph_url=os.environ["EPOCH_GRAPH_URL"],
    backend_url=os.environ["EPOCH_BACKEND_URL"],
    token=_epoch_token,
)

print("Epoch SDK:", importlib.metadata.version("epoch-sdk"))
print("Native SDK:", importlib.metadata.version("stratifyx"))
async with httpx.AsyncClient(
    headers={"Authorization": "Bearer " + _epoch_token}, timeout=20,
) as health:
    for name, origin, route in [
        ("Backend", os.environ["EPOCH_BACKEND_URL"], "/api/v1/health"),
        ("EpochAI", os.environ["EPOCH_GRAPH_URL"], "/health/custom"),
    ]:
        response = await health.get(origin.rstrip("/") + route)
        response.raise_for_status()
        print(name + ": ready")
del _epoch_token

## The request

The task and context are the desk request shown above. The collapsed output
schema declares the requested fields, formats and units; expand it to inspect
or change them. It contains no result values.

Keep the attempt unchanged to reconnect, including after a kernel restart.
Choose a new attempt when intentionally changing the request or refreshing data.

In [ ]:
OUTPUTS = [{'name': 'screener_json',
  'type': 'json',
  'description': 'Identical original shortlist and coverage, serialized as a JSON object.',
  'schema': {'type': 'object',
             'description': 'Saved shortlist with its original snapshot metadata.',
             'properties': {'coverage': {'type': 'object',
                                         'description': 'Exactly the original screening_coverage '
                                                        'row.',
                                         'properties': {'as_of': {'type': 'string',
                                                                  'description': 'Price '
                                                                                 'observation '
                                                                                 'cutoff fixed for '
                                                                                 'this case: '
                                                                                 '2026-10-16.',
                                                                  'format': 'date'},
                                                        'membership_snapshot_at': {'type': 'string',
                                                                                   'description': 'Actual '
                                                                                                  'retrieval '
                                                                                                  'timestamp '
                                                                                                  'of '
                                                                                                  'the '
                                                                                                  'current '
                                                                                                  'S&P '
                                                                                                  '500 '
                                                                                                  'roster; '
                                                                                                  'not '
                                                                                                  'a '
                                                                                                  'claim '
                                                                                                  'of '
                                                                                                  'historical '
                                                                                                  'membership.',
                                                                                   'format': 'date-time'},
                                                        'universe_count': {'type': 'integer',
                                                                           'description': 'Number '
                                                                                          'of '
                                                                                          'unique '
                                                                                          'constituent '
                                                                                          'tickers '
                                                                                          'in the '
                                                                                          'captured '
                                                                                          'roster.'},
                                                        'excluded_missing_data_count': {'type': 'integer',
                                                                                        'description': 'Constituents '
                                                                                                       'excluded '
                                                                                                       'for '
                                                                                                       'missing '
                                                                                                       'required '
                                                                                                       'history, '
                                                                                                       'fields '
                                                                                                       'or '
                                                                                                       'metadata.'},
                                                        'eligible_count': {'type': 'integer',
                                                                           'description': 'Constituents '
                                                                                          'with '
                                                                                          'complete '
                                                                                          'required '
                                                                                          'data '
                                                                                          'and a '
                                                                                          'close '
                                                                                          'strictly '
                                                                                          'above '
                                                                                          'their '
                                                                                          '200-session '
                                                                                          'SMA.'},
                                                        'returned_count': {'type': 'integer',
                                                                           'description': 'min(20, '
                                                                                          'eligible_count); '
                                                                                          'do not '
                                                                                          'pad the '
                                                                                          'shortlist '
                                                                                          'with '
                                                                                          'failing '
                                                                                          'names.'}},
                                         'required': ['as_of',
                                                      'membership_snapshot_at',
                                                      'universe_count',
                                                      'excluded_missing_data_count',
                                                      'eligible_count',
                                                      'returned_count'],
                                         'additionalProperties': False},
                            'shortlist': {'type': 'array',
                                          'description': 'Exactly the original shortlist in '
                                                         'ascending rank order; at most 20 '
                                                         'entries.',
                                          'items': {'type': 'object',
                                                    'description': 'One original shortlist row.',
                                                    'properties': {'rank': {'type': 'integer',
                                                                            'description': 'Sequential '
                                                                                           'rank '
                                                                                           'starting '
                                                                                           'at 1; '
                                                                                           'descending '
                                                                                           'momentum '
                                                                                           'with '
                                                                                           'ticker '
                                                                                           'ascending '
                                                                                           'as the '
                                                                                           'tie-break.'},
                                                                   'ticker': {'type': 'string',
                                                                              'description': 'Unique '
                                                                                             'US-listed '
                                                                                             'constituent '
                                                                                             'ticker; '
                                                                                             'the '
                                                                                             'row '
                                                                                             'label '
                                                                                             'is a '
                                                                                             'security, '
                                                                                             'not '
                                                                                             'a '
                                                                                             'timestamp.'},
                                                                   'company': {'type': 'string',
                                                                               'description': 'Company '
                                                                                              'display '
                                                                                              'name '
                                                                                              'from '
                                                                                              'the '
                                                                                              'resolved '
                                                                                              'asset '
                                                                                              'metadata.'},
                                                                   'sector': {'type': 'string',
                                                                              'description': 'Sector '
                                                                                             'label '
                                                                                             'from '
                                                                                             'the '
                                                                                             'resolved '
                                                                                             'asset '
                                                                                             'metadata.'},
                                                                   'as_of': {'type': 'string',
                                                                             'description': 'Exact '
                                                                                            'completed '
                                                                                            'price '
                                                                                            'session '
                                                                                            'used '
                                                                                            'by '
                                                                                            'the '
                                                                                            'screen: '
                                                                                            '2026-10-16.',
                                                                             'format': 'date'},
                                                                   'momentum_12_1': {'type': 'number',
                                                                                     'description': 'Split-adjusted '
                                                                                                    'close '
                                                                                                    '21 '
                                                                                                    'sessions '
                                                                                                    'ago '
                                                                                                    '/ '
                                                                                                    'split-adjusted '
                                                                                                    'close '
                                                                                                    '252 '
                                                                                                    'sessions '
                                                                                                    'ago '
                                                                                                    '- '
                                                                                                    '1. '
                                                                                                    'Unit: '
                                                                                                    'fraction.'},
                                                                   'return_3m': {'type': 'number',
                                                                                 'description': 'Split-adjusted '
                                                                                                'close '
                                                                                                'at '
                                                                                                'as_of '
                                                                                                '/ '
                                                                                                'close '
                                                                                                '63 '
                                                                                                'sessions '
                                                                                                'ago '
                                                                                                '- '
                                                                                                '1; '
                                                                                                'price '
                                                                                                'return. '
                                                                                                'Unit: '
                                                                                                'fraction.'},
                                                                   'distance_to_sma_200': {'type': 'number',
                                                                                           'description': 'As-of '
                                                                                                          'split-adjusted '
                                                                                                          'close '
                                                                                                          '/ '
                                                                                                          'its '
                                                                                                          'trailing '
                                                                                                          '200-session '
                                                                                                          'SMA '
                                                                                                          '- '
                                                                                                          '1; '
                                                                                                          'strictly '
                                                                                                          'positive. '
                                                                                                          'Unit: '
                                                                                                          'fraction.'},
                                                                   'average_dollar_volume_20': {'type': 'number',
                                                                                                'description': 'Mean '
                                                                                                               'of '
                                                                                                               'actual '
                                                                                                               'quoted '
                                                                                                               'close '
                                                                                                               'times '
                                                                                                               'traded '
                                                                                                               'shares '
                                                                                                               'over '
                                                                                                               'the '
                                                                                                               'last '
                                                                                                               '20 '
                                                                                                               'sessions, '
                                                                                                               'with '
                                                                                                               'price/volume '
                                                                                                               'on '
                                                                                                               'a '
                                                                                                               'consistent '
                                                                                                               'split '
                                                                                                               'basis. '
                                                                                                               'Unit: '
                                                                                                               'USD/day.'}},
                                                    'required': ['rank',
                                                                 'ticker',
                                                                 'company',
                                                                 'sector',
                                                                 'as_of',
                                                                 'momentum_12_1',
                                                                 'return_3m',
                                                                 'distance_to_sma_200',
                                                                 'average_dollar_volume_20'],
                                                    'additionalProperties': False},
                                          'maxItems': 20}},
             'required': ['coverage', 'shortlist'],
             'additionalProperties': False}}]

In [ ]:
TASK = 'Give me that same S&P 500 shortlist and coverage as JSON, keeping the original order and snapshot.'
CONTEXT = ['Use the completed screen linked by the notebook, with the original membership, price cutoff and '
 'exclusions rather than a refreshed screen.']

PREFIX = os.environ.get("EPOCH_NOTEBOOK_PREFIX", "notebook-demo")
BASE_ATTEMPT = "desk-v2-01"
FOLLOWUP_ATTEMPT = "01"
SOURCE_RESEARCH_ID = f"{PREFIX}-sp500_momentum_screener-{BASE_ATTEMPT}-research"
RESEARCH_ID = f"{PREFIX}-sp500_momentum_json-{BASE_ATTEMPT}-{FOLLOWUP_ATTEMPT}-research"

# Read the exact completed base answer; this starts no new work.
source = await epoch.research_status(operation_id=SOURCE_RESEARCH_ID)
assert isinstance(source, ResearchCompleted), "Complete the base notebook first."
ref = source.response
assert ref.study_id and ref.primary_execution_id
source_context = (
    "Authorized saved evidence for this output-only follow-up: "
    f"study_id={ref.study_id}; execution_id={ref.primary_execution_id}; "
    f"source_response_id={ref.response_id}; source_response_version={ref.version}; "
    f"thread_id={ref.thread_id}. Select this exact existing execution as the "
    "primary source. Read its persisted data; create no discovery or execution job."
)
request = RunRequest.model_validate({
    "task": TASK, "context": [*CONTEXT, source_context], "outputs": OUTPUTS,
})
print("Research ID:", RESEARCH_ID)
display(JSON(request.to_wire(), expanded=False))

## Run the session

This is the cell that starts research or reconnects to the same operation. It
shows live activity and the published answer. If Epoch asks a question, choose
an option or write a reply and press **Send reply** inside the cell.

Nothing is selected automatically. If you leave a question waiting, rerun this
cell with the same research ID before continuing to the result cells.

In [ ]:
# The SDK owns streaming, questions and resume.
result = await epoch.run(
    request,
    operation_id=RESEARCH_ID,
    thread_id=source.response.thread_id,
)

## Use the result in Python

The session has already displayed the answer. These cells read its typed parts
for further analysis without printing the same narrative again. Table reads
retrieve every page; JSON and text are already in the saved answer.

In [ ]:
assert isinstance(result, ResearchCompleted), "Resume the session above first."
answer = result.response
json_answers = {part.name: part.value for part in answer.content.outputs if part.type == "json"}
text_answers = {part.name: part.value for part in answer.content.outputs if part.type == "text"}
# Read every page; a page size does not limit the total table size.
tables = {}
for part in result.response.content.outputs:
    if part.type != "table":
        continue
    rows = []
    async for page in epoch.iter_response_table(
        result.response.response_id,
        part.name,
        version=result.response.version,
        page_size=1000,
    ):
        names = [column.name for column in page.columns]
        rows.extend(dict(zip(names, row, strict=True)) for row in page.rows)
    assert len(rows) == part.source.row_count
    tables[part.name] = rows
print("Saved answer:", answer.response_id, "revision", answer.version)
print("Complete table rows:", {name: len(rows) for name, rows in tables.items()})

## Check reuse

The new answer must use the original execution and preserve its answer revision.

In [ ]:
assert answer.primary_execution_id == source.response.primary_execution_id
assert answer.study_id == source.response.study_id
assert answer.response_id != source.response.response_id
original = await epoch.research_status(operation_id=SOURCE_RESEARCH_ID)
assert isinstance(original, ResearchCompleted)
assert original.response == source.response
# Read every page; a page size does not limit the total table size.
source_tables = {}
for part in source.response.content.outputs:
    if part.type != "table":
        continue
    rows = []
    async for page in epoch.iter_response_table(
        source.response.response_id,
        part.name,
        version=source.response.version,
        page_size=1000,
    ):
        names = [column.name for column in page.columns]
        rows.extend(dict(zip(names, row, strict=True)) for row in page.rows)
    assert len(rows) == part.source.row_count
    source_tables[part.name] = rows
value = json_answers["screener_json"]
expected_rows = [{**row, "rank": int(row["rank"])} for row in source_tables["shortlist"]]
expected_coverage = {name: int(value) if name.endswith("count") else value
                     for name, value in source_tables["screening_coverage"][0].items()}
assert value["shortlist"] == expected_rows
assert value["coverage"] == expected_coverage
print("Same source execution; the original saved answer is unchanged.")

## Close this client

Saved work remains on the services. Keep the readable research ID to reconnect.

In [ ]:
await epoch.aclose()